# VibeCore Gemma 2B-IT Inference Server
This notebook loads Gemma 2B-IT in 4-bit nf4 precision, configures an optional LoRA fine-tuning task, spins up a FastAPI backend serving OpenAI-compatible chat completion endpoints, and connects an ngrok tunnel to expose a public proxy URL.

In [ ]:
# Cell 1 - Dependency Installation
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 bitsandbytes==0.43.1 datasets trl fastapi uvicorn[standard] pyngrok nest_asyncio httpx

In [ ]:
# Cell 2 - Core Configuration Parameters
import os

HF_TOKEN = "your_huggingface_token_here"
NGROK_TOKEN = "your_ngrok_token_here"
MODEL_ID = "google/gemma-2b-it"
FINETUNE = False
DATASET_FILE = "my_data.jsonl"

os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# Cell 3 - Load Gemma in 4-Bit nf4 Quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

if HF_TOKEN and HF_TOKEN != "your_huggingface_token_here":
    login(token=HF_TOKEN)

print("Loading tokenizer and model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()
print("Model loaded successfully in 4-bit precision.")

In [ ]:
# Cell 4 - Optional LoRA Fine-Tuning Module
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import json

if FINETUNE:
    print("Preparing dataset and configuration for training...")
    mock_data = [
        {"instruction": "hello", "output": "Hello! I am VibeCore, your cost-optimized assistant. How can I help you save budget today?"},
        {"instruction": "how do you save cost", "output": "I save cost by prompt optimization, exact-match Redis caching, and semantic vector similarity checks."}
    ]
    with open(DATASET_FILE, "w") as f:
        for item in mock_data:
            f.write(json.dumps(item) + "\n")
            
    dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    def formatting_prompts_func(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            text = f"<start_of_turn>user\n{example['instruction'][i]}<end_of_turn>\n<start_of_turn>model\n{example['output'][i]}<end_of_turn>"
            output_texts.append(text)
        return output_texts

    training_args = TrainingArguments(
        output_dir="./gemma-finetuned",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        optim="paged_adamw_8bit",
        logging_steps=10,
        learning_rate=2e-4,
        fp16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="constant"
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=lora_config,
        max_seq_length=512,
        formatting_func=formatting_prompts_func,
        args=training_args
    )
    
    print("Fine-tuning Gemma...")
    trainer.train()
    trainer.model.save_pretrained("./gemma-finetuned")
    print("Fine-tuning completed. Model saved at ./gemma-finetuned")
else:
    print("Fine-tuning skipped. Running model in base evaluation configuration.")

In [ ]:
# Cell 5 - FastAPI Server + ngrok Public Proxy Tunnel
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
import time
import threading
from pyngrok import ngrok
import torch

nest_asyncio.apply()
app = FastAPI(title="VibeCore Gemma Inference Gateway")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 512

@app.get("/health")
def health():
    vram_used = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "healthy",
        "model": MODEL_ID,
        "vram_used_gb": round(vram_used, 2)
    }

@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [
            {
                "id": "gemma-vibecore",
                "object": "model",
                "created": int(time.time()),
                "owned_by": "vibecore"
            }
        ]
    }

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    start_time = time.time()
    try:
        prompt = ""
        for msg in req.messages:
            prompt += f"<start_of_turn>{msg.role}\n{msg.content}<end_of_turn>\n"
        prompt += "<start_of_turn>model\n"
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
        input_tokens = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                  max_new_tokens=req.max_tokens or 512,
                  temperature=req.temperature or 0.7,
                  do_sample=True if (req.temperature or 0.7) > 0.0 else False
            )
        
        generated_ids = outputs[0][input_tokens:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        output_tokens = generated_ids.shape[0]
        
        latency_ms = int((time.time() - start_time) * 1000)
        
        return {
            "id": f"chatcmpl-colab-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": "gemma-vibecore",
            "choices": [
                {
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": response_text
                    },
                    "finish_reason": "stop"
                }
            ],
            "usage": {
                "prompt_tokens": input_tokens,
                "completion_tokens": output_tokens,
                "total_tokens": input_tokens + output_tokens
            },
            "x_latency_ms": latency_ms
        }
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail="OutOfMemoryError. Cuda VRAM cache cleaned.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ngrok execution thread
def run_server():
    if NGROK_TOKEN and NGROK_TOKEN != "your_ngrok_token_here":
        ngrok.set_auth_token(NGROK_TOKEN)
    
    public_url = ngrok.connect(8000).public_url
    print("\n" + "="*60)
    print(f"🚀 VibeCore Gemma server exposed via ngrok!")
    print(f"🔗 Public URL: {public_url}")
    print(f"💡 Copy this URL to your backend .env file as GEMMA_BASE_URL:")
    print(f"   GEMMA_BASE_URL={public_url}")
    print("="*60 + "\n")
    
    uvicorn.run(app, host="0.0.0.0", port=8000)

t = threading.Thread(target=run_server)
t.daemon = True
t.start()

In [ ]:
# Cell 6 - Client Response Check
import httpx
import time

time.sleep(3)
print("Sending test query...")
payload = {
    "model": "gemma-vibecore",
    "messages": [
        {"role": "user", "content": "Explain how a database index speeds up reads in one sentence."}
    ]
}

try:
    response = httpx.post("http://localhost:8000/v1/chat/completions", json=payload, timeout=20.0)
    print("Status Code:", response.status_code)
    print("Response Body:", response.json())
except Exception as e:
    print("Test connection failed:", str(e))

In [ ]:
# Cell 7 - Daemon Ping Keep-Alive Loop
import time
from datetime import datetime
import httpx

print("Starting keep-alive heartbeat loop...")
while True:
    try:
        res = httpx.get("http://localhost:8000/health")
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Heartbeat tick: Status {res.status_code} - VRAM: {res.json().get('vram_used_gb')}GB")
    except Exception as e:
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Heartbeat degraded: {str(e)}")
    time.sleep(60)